# DPO-17: AlpacaEval 2 LC — SimPO Epoch Trajectory

Headline eval for the SimPO comparison. Direct trajectory pairing against
DPO-15's DPO-6 numbers (same SFT init, same dataset, same step counts — only the
loss function differs).

**Two questions this eval answers:**

1. **Does SimPO match DPO on length-controlled win-rate?** If yes (within ±1 pp),
   length-normalization preserved real preference signal while removing the verbosity
   inflation. If LC drops meaningfully (>2 pp), length-normalization destroyed signal.

2. **Does the Raw−LC gap shrink toward zero?** DPO ep3 had LC−Raw = −2.68 pp (length
   bias inflated raw by 2.68 pp). SimPO should have a much smaller gap if its
   length-normalized loss did its job.

**DPO-15 anchors (`weighted_alpaca_eval_gpt4_turbo_new`):**

| Tag | LC | Raw | LC − Raw | Avg len (chars) |
|---|---|---|---|---|
| SFT (zephyr template) | 5.83% | 3.60% | +2.23 pp | 893 |
| DPO ep1 | 5.35% | 6.03% | −0.68 pp | 2,306 |
| DPO ep2 | 8.24% | 9.60% | −1.36 pp | 2,706 |
| **DPO ep3** | **10.82%** | **13.50%** | **−2.68 pp** | **2,678** |
| Zephyr-7B-β (published) | 13.20% | — | — | — |

**Predicted for SimPO** (per DPO-17 planning):
- LC ≈ 10–12% (close to DPO ep3's 10.82%)
- Raw < DPO's 13.50% (less length-bias inflation)
- LC − Raw close to 0 (no verbosity to inflate raw)
- Avg length significantly lower than DPO's 2,678 chars

**Checkpoints:**

| Tag | Checkpoint |
|---|---|
| `simpo_ep1` | `simpo-3ep-dpo17/checkpoint-3732` |
| `simpo_ep2` | `simpo-3ep-dpo17/checkpoint-7464` |
| `simpo_ep3` | `simpo-3ep-dpo17/checkpoint-11196` |

**Cost:** ~\$8–15 per checkpoint × 3 = ~\$24–45
**Time:** ~1.5–2 hr per checkpoint = ~5–6 hr total (in-memory merge ~6 min + gen ~90 min + judge ~15 min)

Same conventions as `ae2_dpo15.ipynb` (in-memory merge to avoid OOM, `weighted_alpaca_eval_gpt4_turbo_new` annotator, `scripts.generation.generate` with role-marker stripping).

In [ ]:
import sys, os, json, subprocess, shutil, tempfile, csv, statistics
from pathlib import Path

REPO_ROOT  = Path("../").resolve()
sys.path.insert(0, str(REPO_ROOT))

BASE_MODEL    = "mistralai/Mistral-7B-v0.1"
CKPT_ROOT     = REPO_ROOT / "checkpoints"
SIMPO_ROOT    = CKPT_ROOT / "simpo-3ep-dpo17"
RUNS_CSV      = REPO_ROOT / "results" / "runs.csv"
AE_OUTPUT_DIR = REPO_ROOT / "results" / "alpaca_eval"
AE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Same annotator as DPO-15 for direct comparability
ANNOTATORS_CONFIG = "weighted_alpaca_eval_gpt4_turbo_new"

# Same generation cap as DPO-15 / DPO-8 — direct length comparability
MAX_NEW_TOKENS = 1024

# SimPO hparams from trainer_state + simpo_qlora.yaml
SIMPO_HPARAMS = dict(stage="simpo", beta=2.0, epochs=3, lr=5e-6, lora_r=128, simpo_gamma=1.0)

CHECKPOINTS = [
    (SIMPO_ROOT / "checkpoint-3732",
     "simpo_ep1", "zephyr-simpo-ep1", "dpo17_simpo_ep1",
     SIMPO_HPARAMS, "DPO-17 AE2 LC: SimPO epoch 1"),
    (SIMPO_ROOT / "checkpoint-7464",
     "simpo_ep2", "zephyr-simpo-ep2", "dpo17_simpo_ep2",
     SIMPO_HPARAMS, "DPO-17 AE2 LC: SimPO epoch 2"),
    (SIMPO_ROOT / "checkpoint-11196",
     "simpo_ep3", "zephyr-simpo-ep3", "dpo17_simpo_ep3",
     SIMPO_HPARAMS, "DPO-17 AE2 LC: SimPO epoch 3"),
]

for ckpt, tag, *_ in CHECKPOINTS:
    status = "OK" if ckpt.exists() else "MISSING"
    print(f"  [{status}] {tag}: {ckpt}")

print(f"\nruns.csv:        {RUNS_CSV}")
print(f"AE output dir:   {AE_OUTPUT_DIR}")
print(f"Annotator:       {ANNOTATORS_CONFIG}")
print(f"max_new_tokens:  {MAX_NEW_TOKENS}")

In [ ]:
assert os.environ.get("OPENAI_API_KEY"), (
    "OPENAI_API_KEY not set.\n"
    "PowerShell: $env:OPENAI_API_KEY = 'sk-...'"
)
print(f"OPENAI_API_KEY set ({len(os.environ['OPENAI_API_KEY'])} chars) ✓")

import alpaca_eval
print(f"alpaca_eval:    v{alpaca_eval.__version__}")

from datasets import load_dataset
AE_DS = load_dataset("tatsu-lab/alpaca_eval", "alpaca_eval", trust_remote_code=True)["eval"]
print(f"AE2 prompts:    {len(AE_DS)} (expected 805)")
print(f"sample columns: {AE_DS.column_names}")
print(f"sample prompt:  {AE_DS[0]['instruction'][:120]}...")

## Helpers — same shape as `ae2_dpo15.ipynb`

`run_one_checkpoint(...)` does: merge LoRA + generate 805 outputs in one in-memory pass (no disk save — avoids the OOM that earlier killed DPO-15 smoke test) → alpaca_eval judge → parse leaderboard → append to `runs.csv`. Idempotent (skips gen/judge if their artifacts already exist).

In [ ]:
CSV_FIELDNAMES = [
    "run_id", "checkpoint", "tag", "stage",
    "beta", "epochs", "lr", "lora_r", "simpo_gamma",
    "max_new_tokens",
    "avg_gen_length", "p90_gen_length",
    "harmful_refusal_rate", "over_refusal_rate", "pref_acc",
    "mt_bench", "alpacaeval2_lc", "notes",
]


def _gen_outputs_path(model_name: str) -> Path:
    return AE_OUTPUT_DIR / f"{model_name}_outputs.json"


def _ae_results_dir() -> Path:
    return AE_OUTPUT_DIR / "results"


def _gen_done(model_name: str, n_required: int = 805) -> bool:
    p = _gen_outputs_path(model_name)
    if not p.exists():
        return False
    try:
        data = json.loads(p.read_text(encoding="utf-8"))
        return isinstance(data, list) and len(data) >= n_required
    except (json.JSONDecodeError, OSError):
        return False


def _judge_done(model_name: str) -> bool:
    lb = _ae_results_dir() / ANNOTATORS_CONFIG / "leaderboard.csv"
    if not lb.exists():
        return False
    import csv as _csv
    with open(lb, newline="", encoding="utf-8") as f:
        for row in _csv.DictReader(f):
            if row.get("") == model_name or row.get("name") == model_name:
                return True
    return False


def merge_and_generate(base_model_id: str, lora_path: str, model_name: str,
                       prompts, output_json: Path,
                       max_new_tokens: int = MAX_NEW_TOKENS):
    """Load base + LoRA, merge in memory, generate outputs — no disk round-trip."""
    import torch, gc
    from peft import PeftModel
    from transformers import AutoModelForCausalLM, AutoTokenizer
    from tqdm.auto import tqdm
    from scripts.generation import generate as _gen

    print(f"  Loading base {base_model_id} (bfloat16, GPU)...")
    tokenizer = AutoTokenizer.from_pretrained(lora_path)
    base = AutoModelForCausalLM.from_pretrained(
        base_model_id, torch_dtype=torch.bfloat16, device_map="auto"
    )
    print(f"  Attaching LoRA from {lora_path}...")
    model = PeftModel.from_pretrained(base, lora_path)
    print("  Merging and unloading (in-memory, no disk save)...")
    model = model.merge_and_unload()
    model.eval()
    free, total = torch.cuda.mem_get_info()
    print(f"  Merged ✓  VRAM: {free/1e9:.1f}/{total/1e9:.1f} GB free")

    results = []
    for p in tqdm(prompts, desc=f"gen {model_name}"):
        response = _gen(
            model, tokenizer,
            [{"role": "user", "content": p["instruction"]}],
            max_new_tokens=max_new_tokens,
        )
        results.append({
            "instruction": p["instruction"],
            "output":      response,
            "generator":   model_name,
            "dataset":     p.get("dataset", "alpaca_eval"),
        })

    output_json.parent.mkdir(parents=True, exist_ok=True)
    with open(output_json, "w", encoding="utf-8") as fout:
        json.dump(results, fout, ensure_ascii=False, indent=2)

    del model, base
    gc.collect()
    torch.cuda.empty_cache()
    print(f"  Wrote {len(results)} outputs → {output_json}")


def run_alpaca_eval(model_outputs: Path, model_name: str):
    from alpaca_eval import evaluate
    print(f"  Running alpaca_eval evaluate against {ANNOTATORS_CONFIG}...")
    df_leaderboard, df_annotations = evaluate(
        model_outputs=str(model_outputs),
        annotators_config=ANNOTATORS_CONFIG,
        name=model_name,
        output_path=str(_ae_results_dir()),
        is_overwrite_leaderboard=False,
        is_return_instead_of_print=True,
    )
    return df_leaderboard


def parse_lc_score(df_leaderboard, model_name: str) -> tuple[float, float, float]:
    if model_name not in df_leaderboard.index:
        raise ValueError(f"{model_name!r} not in leaderboard index: {list(df_leaderboard.index)}")
    row = df_leaderboard.loc[model_name]
    lc  = float(row.get("length_controlled_winrate", row.get("length_controlled_win_rate")))
    raw = float(row.get("win_rate"))
    avg_len = float(row.get("avg_length", -1))
    return round(lc, 2), round(raw, 2), avg_len


def append_csv(row: dict):
    write_header = not RUNS_CSV.exists() or RUNS_CSV.stat().st_size == 0
    if RUNS_CSV.exists() and RUNS_CSV.stat().st_size > 0:
        with open(RUNS_CSV, "rb") as f:
            f.seek(-1, 2)
            last_byte = f.read(1)
        if last_byte not in (b"\n", b"\r"):
            with open(RUNS_CSV, "ab") as f:
                f.write(b"\n")
    with open(RUNS_CSV, "a", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=CSV_FIELDNAMES, extrasaction="ignore")
        if write_header:
            writer.writeheader()
        writer.writerow(row)
    print(f"  Appended to {RUNS_CSV}")


def run_one_checkpoint(ckpt_path: Path, tag: str, model_name: str, run_id: str,
                       hparams: dict, notes: str = "", prompts=None):
    prompts = prompts if prompts is not None else AE_DS
    print(f"\n{'='*64}\n  {tag}  ({ckpt_path.name})  n={len(prompts)}\n{'='*64}")

    outputs_path = _gen_outputs_path(model_name)
    if _gen_done(model_name, n_required=len(prompts)):
        print(f"\n[skip merge+gen] {model_name} already has ≥{len(prompts)} outputs")
    else:
        print(f"\n[1/2] Merge + generate {len(prompts)} outputs for {model_name}")
        merge_and_generate(BASE_MODEL, str(ckpt_path), model_name, prompts, outputs_path)

    if _judge_done(model_name):
        print(f"[skip judge] {model_name} already on leaderboard")
        import pandas as pd
        df = pd.read_csv(_ae_results_dir() / ANNOTATORS_CONFIG / "leaderboard.csv", index_col=0)
    else:
        print(f"\n[2/2] alpaca_eval judge: {model_name}  (~$8–15)")
        df = run_alpaca_eval(outputs_path, model_name)

    lc, raw, avg_len = parse_lc_score(df, model_name)
    print(f"\n  ▶ {tag}  LC win-rate: {lc:.2f}%  |  raw: {raw:.2f}%  |  avg length: {avg_len:.0f}")

    append_csv({
        "run_id":         run_id,
        "checkpoint":     str(ckpt_path),
        "tag":            tag,
        **hparams,
        "max_new_tokens": MAX_NEW_TOKENS,
        "avg_gen_length": round(avg_len, 1),
        "alpacaeval2_lc": lc,
        "notes":          f"{notes} (raw={raw:.2f}%, LC={lc:.2f}%)",
    })
    return lc, raw, avg_len


print("Helpers defined ✓")

## Precheck

Same shape as ae2_dpo15.ipynb's precheck.

In [ ]:
import time, shutil as _shutil

_errors = []

def _check(name, fn):
    try:
        result = fn()
        msg = f" — {result}" if result else ""
        print(f"  ✓ {name}{msg}")
    except Exception as e:
        print(f"  ✗ {name}: {type(e).__name__}: {e}")
        _errors.append(name)


print("=" * 72)
print(" Precheck — environment · checkpoints · API · alpaca_eval · CSV write")
print("=" * 72)


def _check_openai():
    from openai import OpenAI
    OpenAI().models.list()
    return "API key valid"
_check("OPENAI_API_KEY pings OpenAI", _check_openai)


def _check_alpaca_eval():
    import alpaca_eval
    from alpaca_eval import evaluate
    return f"v{alpaca_eval.__version__}, evaluate() importable"
_check("alpaca_eval install + evaluate() importable", _check_alpaca_eval)


def _check_ae_dataset():
    if len(AE_DS) < 800:
        raise RuntimeError(f"AE2 dataset only has {len(AE_DS)} rows (expected ~805)")
    return f"{len(AE_DS)} prompts loaded"
_check("AE2 dataset (tatsu-lab/alpaca_eval) has ~805 prompts", _check_ae_dataset)


def _check_gpu():
    import torch
    if not torch.cuda.is_available():
        raise RuntimeError("CUDA not available")
    free, total = torch.cuda.mem_get_info()
    free_gb, total_gb = free / 1e9, total / 1e9
    if free_gb < 12:
        raise RuntimeError(f"only {free_gb:.1f} GB free of {total_gb:.1f} GB")
    return f"{torch.cuda.get_device_name(0)}, {free_gb:.1f}/{total_gb:.1f} GB free"
_check("GPU available + ≥12 GB free VRAM", _check_gpu)


def _check_disk():
    tmp = Path(tempfile.gettempdir())
    free_gb = _shutil.disk_usage(tmp).free / 1e9
    if free_gb < 30:
        raise RuntimeError(f"only {free_gb:.1f} GB free at {tmp}")
    return f"{free_gb:.1f} GB free at {tmp}"
_check("Temp disk ≥30 GB free (merged models ~14 GB each)", _check_disk)


def _check_checkpoints():
    for ckpt, *_ in CHECKPOINTS:
        if not ckpt.exists():
            raise FileNotFoundError(ckpt)
        if not (ckpt / "adapter_config.json").exists():
            raise FileNotFoundError(f"{ckpt}/adapter_config.json")
    return f"{len(CHECKPOINTS)} checkpoints OK"
_check("All checkpoint paths + adapter_config.json exist", _check_checkpoints)


def _check_chat_templates():
    from transformers import AutoTokenizer
    for ckpt, tag, *_ in CHECKPOINTS:
        ct = AutoTokenizer.from_pretrained(str(ckpt)).chat_template or ""
        if "<|user|>" not in ct or "<|assistant|>" not in ct:
            raise RuntimeError(f"{tag}: tokenizer chat_template missing Zephyr markers")
    return f"{len(CHECKPOINTS)} tokenizers carry Zephyr template"
_check("Tokenizers have Zephyr chat_template baked in", _check_chat_templates)


def _check_csv_write():
    sentinel_id = f"__precheck_ae_{int(time.time())}__"
    backup = RUNS_CSV.read_bytes() if RUNS_CSV.exists() else None
    try:
        append_csv({
            "run_id": sentinel_id, "checkpoint": "precheck", "tag": "precheck",
            "stage": "simpo", "beta": 2.0, "epochs": 3, "lr": 5e-6, "lora_r": 128,
            "simpo_gamma": 1.0, "alpacaeval2_lc": 0.0,
            "notes": "precheck sentinel — should be rolled back",
        })
        with open(RUNS_CSV, newline="") as f:
            reader = csv.DictReader(f)
            cols = reader.fieldnames
            rows = [r for r in reader if r["run_id"] == sentinel_id]
        if len(rows) != 1:
            raise RuntimeError(f"sentinel not found after append (got {len(rows)} rows)")
        if rows[0]["alpacaeval2_lc"] != "0.0" or rows[0]["tag"] != "precheck":
            raise RuntimeError(f"column misalignment: {rows[0]}")
        if cols != CSV_FIELDNAMES:
            raise RuntimeError(f"header drift — file has {cols}, code expects {CSV_FIELDNAMES}")
        return f"{len(cols)} columns aligned, sentinel rolled back"
    finally:
        if backup is not None:
            RUNS_CSV.write_bytes(backup)
        else:
            RUNS_CSV.unlink(missing_ok=True)
_check("runs.csv append + alignment + rollback", _check_csv_write)


print("=" * 72)
if _errors:
    print(f"  ✗ {len(_errors)} precheck(s) failed: {_errors}")
    raise SystemExit(f"Precheck failed: {_errors}")
else:
    print("  ✓ All prechecks passed — safe to launch")
print("=" * 72)

## 1. AE2 LC — SimPO epoch 1 (`checkpoint-3732`)

In [ ]:
_ckpt, _tag, _mid, _rid, _hp, _notes = CHECKPOINTS[0]
simpo_ep1_lc, simpo_ep1_raw, simpo_ep1_avglen = run_one_checkpoint(_ckpt, _tag, _mid, _rid, _hp, _notes)

## 2. AE2 LC — SimPO epoch 2 (`checkpoint-7464`)

In [ ]:
_ckpt, _tag, _mid, _rid, _hp, _notes = CHECKPOINTS[1]
simpo_ep2_lc, simpo_ep2_raw, simpo_ep2_avglen = run_one_checkpoint(_ckpt, _tag, _mid, _rid, _hp, _notes)

## 3. AE2 LC — SimPO epoch 3 (`checkpoint-11196`)

Headline checkpoint. This is the SimPO-vs-DPO comparison row for the writeup.

In [ ]:
_ckpt, _tag, _mid, _rid, _hp, _notes = CHECKPOINTS[2]
simpo_ep3_lc, simpo_ep3_raw, simpo_ep3_avglen = run_one_checkpoint(_ckpt, _tag, _mid, _rid, _hp, _notes)

## 4. Results — SimPO vs DPO AE2 LC trajectory + length-bias decomposition

The headline comparison. Two things to look at:

1. **LC win-rate** — does SimPO match DPO's preference quality?
2. **LC − Raw gap** — does the length-bias contribution shrink as predicted?

In [ ]:
# DPO-15 anchors (LC, Raw, avg_length_chars)
AE2_DPO = {
    "sft_zephyr": (5.83,  3.60,  893),
    "dpo6_ep1":   (5.35,  6.03,  2306),
    "dpo6_ep2":   (8.24,  9.60,  2706),
    "dpo6_ep3":   (10.82, 13.50, 2678),
}
AE2_SIMPO = {
    "simpo_ep1": (simpo_ep1_lc, simpo_ep1_raw, simpo_ep1_avglen),
    "simpo_ep2": (simpo_ep2_lc, simpo_ep2_raw, simpo_ep2_avglen),
    "simpo_ep3": (simpo_ep3_lc, simpo_ep3_raw, simpo_ep3_avglen),
}

print("=" * 100)
print(" AE2 LC: DPO vs SimPO trajectory (DPO-17)")
print("=" * 100)
print(f"{'Model':<30} {'LC':>8} {'Raw':>8} {'LC−Raw':>10} {'AvgLen(ch)':>12}")
print("-" * 100)
for label, key, source in [
    ("SFT (anchor)",   "sft_zephyr", AE2_DPO),
    ("DPO ep1",        "dpo6_ep1",   AE2_DPO),
    ("DPO ep2",        "dpo6_ep2",   AE2_DPO),
    ("DPO ep3",        "dpo6_ep3",   AE2_DPO),
    ("SimPO ep1",      "simpo_ep1", AE2_SIMPO),
    ("SimPO ep2",      "simpo_ep2", AE2_SIMPO),
    ("SimPO ep3",      "simpo_ep3", AE2_SIMPO),
]:
    lc, raw, alen = source[key]
    print(f"{label:<30} {lc:>7.2f}% {raw:>7.2f}% {lc-raw:>+9.2f} {alen:>12.0f}")
print(f"{'Zephyr-7B-β (published)':<30} {'13.20%':>8} {'—':>8} {'—':>10} {'—':>12}")
print("=" * 100)

# Headline comparison: DPO ep3 vs SimPO ep3
dpo_lc, dpo_raw, dpo_len = AE2_DPO["dpo6_ep3"]
spo_lc, spo_raw, spo_len = AE2_SIMPO["simpo_ep3"]

print("\nHeadline: DPO ep3 vs SimPO ep3 (same SFT init, same data, same step count, loss differs):")
print(f"  LC win-rate     : {dpo_lc:.2f}% → {spo_lc:.2f}%   ({spo_lc - dpo_lc:+.2f} pp)")
print(f"  Raw win-rate    : {dpo_raw:.2f}% → {spo_raw:.2f}%   ({spo_raw - dpo_raw:+.2f} pp)")
print(f"  LC − Raw gap    : {dpo_lc - dpo_raw:+.2f} pp → {spo_lc - spo_raw:+.2f} pp  (length contribution change: {(spo_lc - spo_raw) - (dpo_lc - dpo_raw):+.2f})")
print(f"  Avg gen length  : {dpo_len:.0f} chars → {spo_len:.0f} chars  ({100*(spo_len-dpo_len)/dpo_len:+.1f}%)")

print("\nPrediction check:")
if abs(spo_lc - dpo_lc) <= 1.5:
    print("  ✓ LC matched within ±1.5 pp — length-normalization preserved preference signal")
elif spo_lc < dpo_lc - 1.5:
    print(f"  ⚠ LC dropped by {dpo_lc - spo_lc:.2f} pp — length-normalization destroyed some signal")
else:
    print(f"  ✓ LC improved by {spo_lc - dpo_lc:.2f} pp — SimPO outperformed DPO under length control")

lc_raw_gap_simpo  = spo_lc - spo_raw
lc_raw_gap_dpo    = dpo_lc - dpo_raw
if abs(lc_raw_gap_simpo) < 1.0:
    print(f"  ✓ SimPO LC≈Raw (gap {lc_raw_gap_simpo:+.2f} pp) — length-bias contribution near zero")
    print(f"    DPO ep3 had a {lc_raw_gap_dpo:+.2f} pp length contribution; SimPO removed it via length-normalized loss")
elif lc_raw_gap_simpo < lc_raw_gap_dpo:
    print(f"  ✓ Length contribution shrunk from {lc_raw_gap_dpo:+.2f} pp to {lc_raw_gap_simpo:+.2f} pp")
else:
    print(f"  ⚠ Length contribution did NOT shrink (DPO {lc_raw_gap_dpo:+.2f} pp → SimPO {lc_raw_gap_simpo:+.2f} pp)")
    print("    Check that cpo_alpha=0.0 was applied during training (pure SimPO, not hybrid).")

if spo_len < dpo_len * 0.7:
    print(f"  ✓ SimPO outputs are {100*(1-spo_len/dpo_len):.0f}% shorter than DPO — verbosity inflation removed")

print("\nNext: write the dpo17_simpo_review.md following the dpo15 pattern.")